In [0]:
import yaml

with open("/Workspace/Users/roksolana.shendiu770@softserve.academy/petroleum-consumption-pipeline/lab4/prod/pipeline_config.yaml") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
bronze_schema = config["bronze_schema"]
silver_schema = config["silver_schema"]

source_table = config["consumption"]["source_table"]
target_table = config["consumption"]["target_table"]

bronze_full_name = f"{catalog}.{bronze_schema}.{source_table}"
silver_full_name = f"{catalog}.{silver_schema}.{target_table}"

In [0]:
import logging
from pyspark.sql.functions import (
    col, row_number, current_timestamp, sha2, concat_ws, expr, lit
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logger = logging.getLogger("silver_consumption_pipeline")
logger.setLevel(logging.INFO)

try:
    bronze_df = spark.table(bronze_full_name)

    cleaned_df = (bronze_df
        .withColumnRenamed("series", "series_bk")
        .withColumnRenamed("duoarea", "duoarea_bk")
        .withColumnRenamed("product", "product_bk")
        .withColumn("period_bk", expr("try_cast(period as date)"))
        .withColumn("consumption_value", expr("try_cast(value as decimal(10,3))"))
        .select(
            "series_bk", "duoarea_bk", "period_bk", "product_bk",
            "units", "consumption_value", "source_filename", "ingestion_timestamp"
        )
    )

    w = Window.partitionBy("series_bk", "duoarea_bk", "period_bk") \
              .orderBy(col("ingestion_timestamp").desc())

    deduped_df = (cleaned_df
        .withColumn("rn", row_number().over(w))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    final_df = deduped_df.withColumn(
        "consumption_sk",
        sha2(concat_ws("||",
            col("series_bk"),
            col("duoarea_bk"),
            col("period_bk").cast("string")
        ), 256)
    ).withColumn(
        "_source_system", lit("EIA_petroleum_consumption")
    ).withColumn(
        "_ingested_at", current_timestamp()
    ).withColumn(
        "_updated_at", lit(None).cast("timestamp")
    ).select(
        "consumption_sk", "series_bk", "duoarea_bk", "period_bk", "product_bk",
        "units", "consumption_value", "_source_system", "_ingested_at", "_updated_at"
    )

    target_table_obj = DeltaTable.forName(spark, silver_full_name)

    merge_result = (target_table_obj.alias("t")
        .merge(final_df.alias("s"), "t.consumption_sk = s.consumption_sk")
        .whenMatchedUpdate(
            condition="t.consumption_value <> s.consumption_value",
            set={
                "consumption_value": "s.consumption_value",
                "_source_system": "s._source_system",
                "_updated_at": "current_timestamp()"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    stats = merge_result.collect()[0].asDict()
    logger.info(f"MERGE completed: {stats}")

    if stats["num_affected_rows"] == 0 and bronze_df.isEmpty():
        raise ValueError(f"Source table {bronze_full_name} is empty, aborting pipeline")

except Exception as e:
    logger.error(f"Pipeline failed: {e}")
    raise